In [ ]:
import cv2
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as patches

%matplotlib notebook
import random

In [ ]:
# Datos conocidos
image_width, image_height = 1920, 1080
fov = 90  # grados
camera_height = 1.3  # metros

# Cargar imagen y líneas
img = cv2.imread('vp.jpg')
lines_near_vps = np.load('merged_lines_near_vps.npy')

# Copias para gráficos
img_lines = img.copy()
img_reprojected = img.copy()
ground_lines = []

# Calcular focal en píxeles
f = image_width / (2 * np.tan(np.radians(fov / 2)))
cx, cy = image_width / 2, image_height / 2

# Matriz intrínseca
K = np.array([
    [f, 0, cx],
    [0, f, cy],
    [0, 0, 1]
])
K_inv = np.linalg.inv(K)

for line in lines_near_vps:
    x1, y1, x2, y2 = line[0]

    # Dibujar línea original en rojo
    cv2.line(img_lines, (x1, y1), (x2, y2), (0, 0, 255), 1)

    # Proyección al espacio
    p1_img = np.array([x1, y1, 1])
    p2_img = np.array([x2, y2, 1])

    # Rayos en cámara
    r1_cam = K_inv @ p1_img
    r2_cam = K_inv @ p2_img

    # Escalar rayos para intersectar el plano Z=0 (suelo)
    scale1 = camera_height / r1_cam[1]
    scale2 = camera_height / r2_cam[1]

    p1_ground = r1_cam * scale1
    p2_ground = r2_cam * scale2

    ground_lines.append((p1_ground, p2_ground))

    # Reproyección a imagen
    p1_proj = K @ p1_ground
    p1_proj /= p1_proj[2]

    p2_proj = K @ p2_ground
    p2_proj /= p2_proj[2]

    # Dibujar línea reproyectada en verde
    pt1_proj = (int(p1_proj[0]), int(p1_proj[1]))
    pt2_proj = (int(p2_proj[0]), int(p2_proj[1]))
    cv2.line(img_reprojected, pt1_proj, pt2_proj, (0, 255, 0), 1)

In [ ]:

# === Graficar ===
# Imagen original con líneas rojas
fig1 = plt.figure(figsize=(6, 6))
plt.imshow(cv2.cvtColor(img_lines, cv2.COLOR_BGR2RGB))
plt.title("1. Líneas originales en imagen")
plt.axis("off")
plt.show()

In [ ]:

# Plano del suelo (top view)
fig2 = plt.figure(figsize=(6, 6))
for p1, p2 in ground_lines:
    plt.plot([p1[0], p2[0]], [p1[2], p2[2]], 'b-')  # Ejes X-Z

# Dibujar el rectángulo azul (carrito)
car_x, car_z = -1, -4  # Coordenadas iniciales del carrito
car_width, car_length = 2, 4  # Ancho y largo del carrito
rect = patches.Rectangle((car_x, car_z), car_width, car_length, linewidth=1, edgecolor='blue', facecolor='blue')
plt.gca().add_patch(rect)

plt.title("2. Líneas proyectadas al espacio (plano suelo)")
plt.xlabel("X (m)")
plt.ylabel("Y (m)")
plt.axis("equal")
plt.grid(True)
plt.xlim(-10, 10)  # Centrado en x=0 con escala 5
plt.ylim(-10, 30)   # Centrado en y=10 con escala 5
plt.show()

In [ ]:

# Imagen con líneas reproyectadas (verde)
fig3 = plt.figure(figsize=(6, 6))
plt.imshow(cv2.cvtColor(img_reprojected, cv2.COLOR_BGR2RGB))
plt.title("3. Líneas reproyectadas desde el plano")
plt.axis("off")
plt.show()


In [ ]:
from shapely.geometry import LineString, Point
import matplotlib.patches as patches

LD = 20  # Límite de distancia en metros
camera_pos = np.array([0, camera_height, 0])  # cámara en 3D
intersections = []

# Construir segmentos 2D sobre X-Z
lines_2d = []
for p1, p2 in ground_lines:
    line = LineString([(p1[0], p1[2]), (p2[0], p2[2])])  # (x, z)
    lines_2d.append(line)

# Calcular todas las intersecciones únicas entre pares de líneas
for i in range(len(lines_2d)):
    for j in range(i + 1, len(lines_2d)):
        inter = lines_2d[i].intersection(lines_2d[j])
        if isinstance(inter, Point):
            ix, iz = inter.x, inter.y
            inter_point_3d = np.array([ix, 0, iz])
            dist = np.linalg.norm(inter_point_3d - camera_pos)
            if dist < LD:
                intersections.append((ix, iz, dist))

# === Graficar plano del suelo con intersecciones ===
fig2 = plt.figure(figsize=(6, 6))
ax2 = plt.gca()

# Dibujar líneas proyectadas al suelo
for p1, p2 in ground_lines:
    ax2.plot([p1[0], p2[0]], [p1[2], p2[2]], 'b-')  # (x, z)

# Dibujar el carrito
rect = patches.Rectangle((car_x, car_z), car_width, car_length,
                         linewidth=1, edgecolor='blue', facecolor='blue')
ax2.add_patch(rect)

# Dibujar intersecciones y vectores
for ix, iz, dist in intersections:
    ax2.annotate('', xy=(ix, iz), xytext=(0, 0),
                 arrowprops=dict(arrowstyle='->', color='red', lw=1))
    ax2.text(ix, iz, f"{dist:.2f}m", fontsize=7, color='darkred')

# Configuración de gráfico
ax2.set_title(f"2. Líneas proyectadas e intersecciones (< {LD} m)")
ax2.set_xlabel("X (m)")
ax2.set_ylabel("Z (m)")
ax2.set_aspect("equal")
ax2.grid(True)
ax2.set_xlim(-20, 20)
ax2.set_ylim(-10, 30)
plt.show()


In [ ]:
# Ejemplo: extender 5 líneas paralelas a la dirección promedio



# Suponiendo que tienes una línea base
base_p1, base_p2 = ground_lines[0]
v = base_p2 - base_p1  # vector de dirección
v /= np.linalg.norm(v)  # normalizado

# Vector normal en el plano XZ (para separación entre líneas)
n = np.array([-v[2], 0, v[0]])  # perpendicular en X-Z
n /= np.linalg.norm(n)

# Distancia estimada entre líneas (puedes mejorar esto con media entre líneas existentes)
d = 2.5  # metros

# Generar y graficar nuevas líneas paralelas
fig = plt.figure(figsize=(6, 6))
for offset in range(-3, 4):  # -3 a +3 líneas
    shift = offset * d * n
    new_p1 = base_p1 + shift
    new_p2 = base_p2 + shift
    plt.plot([new_p1[0], new_p2[0]], [new_p1[2], new_p2[2]], 'g--')

# Las líneas detectadas originales
for p1, p2 in ground_lines:
    plt.plot([p1[0], p2[0]], [p1[2], p2[2]], 'b-')

plt.title("Extensión de líneas en el plano del suelo")
plt.xlabel("X (m)")
plt.ylabel("Z (m)")
plt.axis("equal")
plt.grid(True)
plt.show()


In [ ]:
def def_grid_lines(r0,r1,c0,c1,w,h):
    R=np.linspace(r0,r1,r1-r0+1)
    C=np.linspace(c0,c1,c1-c0+1)
    n = len(R)+len(C)
    l=np.zeros((3, n))

    #Definimos primero lineas horizontales
    idx=0
    for i in R:
        l[:,idx]=[0, 1, h*i]
        idx +=1

    #Definimos primero lineas verticales
    for i in C:
        l[:,idx]=[1, 0, h*i]
        idx=idx+1
    return l

grid_lines = def_grid_lines(-grid_size, grid_size, -grid_size, grid_size, 1, 1)



In [ ]:
from clipLine import *
import time


def line_similarity(line_a, line_b, threshold=1, normType = 0, region=[1920, 1080]):
    distance = np.inf
    if normType == 1:
        # Normalize the lines as homogeneous variable
        linea_n = line_a / line_a[2]
        lineb_n = line_b / line_b[2]
        tmp = linea_n[:2] - lineb_n[:2]
        distance = np.dot(tmp, tmp)
    elif normType == 2:
        # Normalize the lines according to their size
        linea_n = line_b / np.linalg.norm(line_b)
        linea_n = line_a / np.linalg.norm(line_a)
        tmp = lineb_n - linea_n
        distance = np.dot(tmp, tmp)
    else:
        
        pl1, pl2, success = clipLine(line_a, (0,region[1]//2), (region[0],region[1]//2))
        
        if success == True:
            pl3, pl4, success = clipLine(line_b, (0,region[1]//2), (region[0],region[1]//2))
            
            if success == True:
                d=[]
                tmp = pl1[:2]-pl3[:2]
                d.append(np.dot(tmp,tmp)) #Squared Distance between pl1 and pl2
                tmp = pl2[:2]-pl4[:2]
                d.append(np.dot(tmp,tmp)) #Squared Distance between pl2 and pl4
                tmp = pl2[:2]-pl3[:2]
                d.append(np.dot(tmp,tmp)) #Squared Distance between pl2 and pl3
                tmp = pl1[:2]-pl4[:2]
                d.append(np.dot(tmp,tmp)) #Squared Distance between pl1 and pl4
                tmp = pl1[:2]-pl2[:2]
                distance = min(d)
        

    # print('distance = ', distance)

    # Compute similarity
    return distance <= (threshold * threshold), distance


l = lines_near_vps.copy().astype('float64')
n,_,_ = l.shape

figA, axA = plt.subplots()

n = 5
print("n=",n)

for i in range(n):
    axA.plot([l[i,0,0], l[i,0,2]],[l[i,0,1], l[i,0,3]],linewidth=3)
    lhm = np.cross(np.array([l[i,0,0],l[i,0,1],1]),np.array([l[i,0,2],l[i,0,3],1]))
    lhm = lhm.astype('float64')
    lhm /= lhm[2]
    L1A, L1B, Success = clipLine(lhm, (0,540),(1920,540))
    axA.plot([L1A[0],L1B[0]],[L1A[1],L1B[1]], linewidth=1)
 
figB, axB= plt.subplots()

D=[]
for i in range(n-1):
    lhmA = np.cross(np.array([l[i,0,0],l[i,0,1],1]),np.array([l[i,0,2],l[i,0,3],1]))
    axB.plot([l[i,0,0], l[i,0,2]],[l[i,0,1], l[i,0,3]],'r')
    for j in range(i+1, n):
        axB.plot([l[j,0,0], l[j,0,2]],[l[j,0,1], l[j,0,3]])
        lhmB = np.cross(np.array([l[j,0,0],l[j,0,1],1]),np.array([l[j,0,2],l[j,0,3],1]))
        similar, d = line_similarity(lhmA, lhmB)
        print ("D(%d,%d)=%f" %(i, j, d))
        D.append(d)
        
      
    
axA.plot([0,1920,1920,0,0],[540,540,1080,1080,540],'k', linewidth=3)
print(np.sqrt(D))